# IAN 601 — Module 11: Advanced Pandas

Advanced Pandas is less about collecting more methods and more about building clear, reusable data workflows.

This standalone notebook focuses on reshaping, grouping, time-based analysis, safer joins, and method chaining using a sales CSV stored in the course repository.

Dataset: `IAN601_Module11_Advanced_Pandas_Sales.csv`

## Learning goals

By the end of this notebook, you should be able to:

1. build readable method chains;
2. filter and create columns with `query()` and `assign()`;
3. perform named aggregations with `groupby()`;
4. explain the difference between `agg()` and `transform()`;
5. create pivot tables;
6. reshape wide data with `melt()`;
7. expand list-like values with `explode()`;
8. work with datetime columns;
9. resample time-based data;
10. calculate rolling summaries;
11. create categories with mapping and binning;
12. validate joins with `merge()`;
13. build reusable workflows with `pipe()`.

## How to use this notebook

Run the cells from top to bottom the first time so that variables and imported libraries are available when later cells need them.

As you work:

1. read the explanation before the code;
2. predict the result;
3. run the cell;
4. inspect the output carefully;
5. change one value, condition, or option;
6. rerun the cell and explain what changed.

For notebooks that load a CSV from GitHub, the dataset is loaded automatically. If you restart the runtime, run the notebook again from the beginning.

### Expandable syntax guides

Each major section includes a collapsed **Syntax & parameters** box. Open it when you want more detail about function signatures, important parameters, related methods, or common usage patterns. You can keep it closed when you want to focus only on the main workflow.

## 1. Load the CSV dataset

A Pandas workflow normally begins by loading data and immediately checking what arrived.

Here we read a CSV directly from GitHub, store it in a DataFrame named `orders`, print its shape, and inspect the first few rows.

A useful loading habit is:

1. read the file;
2. check the number of rows and columns;
3. inspect column names and data types;
4. look at a few rows before doing any transformation.

The dataset contains 300 sales transactions with dates, customers, regions, channels, products, prices, discounts, sales amounts, and tags.

<details>
<summary><strong>Syntax & parameters — read_csv()</strong></summary>

<p><strong>Basic syntax</strong></p>
<pre><code class="language-python">pd.read_csv(filepath_or_buffer, ...)</code></pre>

<p><strong>Common parameters</strong></p>
<ul>
<li><code>filepath_or_buffer</code>: local path, URL, or file-like object.</li>
<li><code>sep</code>: column separator. Default is a comma.</li>
<li><code>usecols</code>: load only selected columns.</li>
<li><code>dtype</code>: force specific column types.</li>
<li><code>parse_dates</code>: parse selected columns as dates while loading.</li>
<li><code>na_values</code>: additional strings that should be treated as missing.</li>
<li><code>nrows</code>: load only the first N rows.</li>
<li><code>encoding</code>: text encoding such as <code>"utf-8"</code>.</li>
</ul>

<p><strong>Useful inspection methods after loading</strong></p>
<pre><code class="language-python">df.head()
df.tail()
df.shape
df.columns
df.dtypes
df.info()
df.sample(5)</code></pre>

<p><strong>Tip:</strong> Loading successfully does not guarantee the data types are correct. Always inspect the result.</p>
</details>

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)

csv_url = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-601-Introduction-to-Data-Analytics-Methods-and-Approaches/"
    "main/IAN601_Module11_Advanced_Pandas_Sales.csv"
)

orders = pd.read_csv(csv_url)

print("Dataset shape:", orders.shape)
orders.head()

## 2. Method chaining

**Method chaining** means applying several DataFrame operations in sequence.

Instead of creating many temporary variables, each method returns a DataFrame that becomes the input to the next method.

Read the example from top to bottom:

1. start with `orders`;
2. keep rows where `amount >= 70`;
3. create a new `fee` column;
4. sort from largest to smallest amount.

Parentheses let us place each operation on its own line without using backslashes. This style is common in analytical Python because the workflow reads like a sequence of steps.

<details>
<summary><strong>Syntax & parameters — method chaining</strong></summary>

<p><strong>General pattern</strong></p>
<pre><code class="language-python">result = (
    df
    .method_one(...)
    .method_two(...)
    .method_three(...)
)</code></pre>

<p>Each method receives the result returned by the previous method.</p>

<p><strong>Methods commonly used in chains</strong></p>
<ul>
<li><code>query()</code> — filter rows.</li>
<li><code>assign()</code> — create columns.</li>
<li><code>sort_values()</code> — sort rows.</li>
<li><code>drop()</code> — remove columns or rows.</li>
<li><code>rename()</code> — rename columns.</li>
<li><code>groupby()</code> + <code>agg()</code> — summarize groups.</li>
<li><code>reset_index()</code> — move index levels back into columns.</li>
<li><code>pipe()</code> — call a custom function inside the chain.</li>
</ul>

<p><strong>sort_values() example</strong></p>
<pre><code class="language-python">df.sort_values(
    by=["region", "amount"],
    ascending=[True, False],
    na_position="last"
)</code></pre>

<p><code>by</code> chooses the sort columns. <code>ascending</code> may be one Boolean or a list of Booleans.</p>
</details>

In [ ]:
result = (
    orders
    .query("amount >= 70")
    .assign(fee=lambda df: df["amount"] * 0.05)
    .sort_values("amount", ascending=False)
)

result[["order_id", "amount", "fee"]]

### Try it
Change the filter from `amount >= 70` to another value. Then change the fee from 5% to 8%.

## 3. `query()` and `assign()`

`query()` filters rows using an expression written as a string.

`assign()` creates one or more columns and returns a **new DataFrame**, which makes it especially useful inside a chain.

In the example below:

- `query()` keeps Web orders with an amount of at least 50;
- `assign()` calculates tax as 7% of the order amount.

A lambda inside `assign()` receives the DataFrame at that point in the chain, so it can refer to columns that already exist or were created earlier in the same workflow.

<details>
<summary><strong>Syntax & parameters — query() and assign()</strong></summary>

<p><strong>query()</strong></p>
<pre><code class="language-python">df.query("amount >= 50 and channel == 'Web'")</code></pre>

<ul>
<li>Use <code>and</code>, <code>or</code>, and <code>not</code> inside the query string.</li>
<li>Use <code>@variable_name</code> to reference a Python variable outside the DataFrame.</li>
</ul>

<pre><code class="language-python">minimum = 50
df.query("amount >= @minimum")</code></pre>

<p><strong>assign()</strong></p>
<pre><code class="language-python">df.assign(
    tax=lambda x: x["amount"] * 0.07,
    total=lambda x: x["amount"] + x["tax"]
)</code></pre>

<p><code>assign()</code> returns a new DataFrame. When several columns are created, later expressions can use columns created earlier in the same <code>assign()</code> call.</p>

<p><strong>Related filtering syntax</strong></p>
<pre><code class="language-python">df.loc[df["amount"] >= 50]
df[df["amount"] >= 50]</code></pre>

<p><code>query()</code> is often easier to read; Boolean indexing is often more flexible for complex Python expressions.</p>
</details>

In [ ]:
web_orders = (
    orders
    .query("channel == 'Web' and amount >= 50")
    .assign(tax=lambda df: df["amount"] * 0.07)
)

web_orders[["order_id", "channel", "amount", "tax"]]

## 4. Named aggregation with `groupby()`

`groupby()` splits rows into groups based on one or more keys.

`agg()` then calculates one or more summary statistics for each group.

**Named aggregation** lets us choose meaningful output-column names such as `total_sales`, `average_order`, and `orders`.

This is often easier to read than calculating each statistic separately.

<details>
<summary><strong>Syntax & parameters — groupby() and named agg()</strong></summary>

<p><strong>Basic grouping</strong></p>
<pre><code class="language-python">df.groupby("region")["amount"].sum()</code></pre>

<p><strong>Named aggregation</strong></p>
<pre><code class="language-python">df.groupby("region", as_index=False).agg(
    total_sales=("amount", "sum"),
    average_order=("amount", "mean"),
    orders=("order_id", "count")
)</code></pre>

<p>Each named aggregation has the form:</p>
<pre><code>output_name=("source_column", "aggregation")</code></pre>

<p><strong>Useful groupby() parameters</strong></p>
<ul>
<li><code>by</code>: one column or a list of grouping columns.</li>
<li><code>as_index=False</code>: keep grouping keys as normal columns.</li>
<li><code>dropna=False</code>: keep missing group labels as a group.</li>
<li><code>sort=False</code>: avoid automatically sorting group keys.</li>
</ul>

<p><strong>Common aggregations</strong>: <code>sum</code>, <code>mean</code>, <code>median</code>, <code>min</code>, <code>max</code>, <code>count</code>, <code>size</code>, <code>nunique</code>, <code>std</code>.</p>

<p><code>count</code> ignores missing values in the selected column; <code>size</code> counts rows.</p>
</details>

In [ ]:
region_summary = (
    orders
    .groupby("region", as_index=False)
    .agg(
        total_sales=("amount", "sum"),
        average_order=("amount", "mean"),
        orders=("order_id", "count")
    )
)

region_summary.round(2)

## 5. `agg()` versus `transform()`

Both `agg()` and `transform()` can calculate statistics within groups, but they return different shapes.

- `agg()` **reduces** each group to summary rows.
- `transform()` returns a result aligned with the **original rows**.

That makes `transform()` especially useful when each original row needs to be compared with its group statistic.

Here, every order receives its region's average order amount, and then we calculate how far the individual order is above or below that average.

<details>
<summary><strong>Syntax & parameters — agg() vs transform()</strong></summary>

<p><strong>Aggregation reduces rows</strong></p>
<pre><code class="language-python">df.groupby("region")["amount"].agg("mean")</code></pre>

<p><strong>Transform preserves row alignment</strong></p>
<pre><code class="language-python">df.groupby("region")["amount"].transform("mean")</code></pre>

<p>The transformed result has the same length as the original DataFrame, so it can be assigned directly as a new column.</p>

<p><strong>Custom transform</strong></p>
<pre><code class="language-python">df["centered"] = (
    df.groupby("region")["amount"]
      .transform(lambda s: s - s.mean())
)</code></pre>

<p><strong>Rule of thumb</strong></p>
<ul>
<li>Need one row per group? Use <code>agg()</code>.</li>
<li>Need a group statistic repeated beside every original row? Use <code>transform()</code>.</li>
</ul>

<p>Related methods include <code>filter()</code> for keeping or removing whole groups and <code>apply()</code> for more general group operations.</p>
</details>

In [ ]:
orders_compare = orders.copy()

orders_compare["region_average"] = (
    orders_compare
    .groupby("region")["amount"]
    .transform("mean")
)

orders_compare["difference"] = (
    orders_compare["amount"] - orders_compare["region_average"]
)

orders_compare[
    ["order_id", "region", "amount", "region_average", "difference"]
].round(2)

Notice that `transform()` did **not** reduce the number of rows. Each order still appears once.

## 6. Pivot tables

A pivot table summarizes data across two or more dimensions.

In this example:

- `region` becomes the row index;
- `channel` becomes the columns;
- `amount` is the numeric value being summarized;
- `sum` is the aggregation.

The result is a compact matrix showing total sales for each region/channel combination.

<details>
<summary><strong>Syntax & parameters — pivot_table()</strong></summary>

<pre><code class="language-python">pd.pivot_table(
    data,
    index="region",
    columns="channel",
    values="amount",
    aggfunc="sum",
    fill_value=0
)</code></pre>

<p><strong>Main parameters</strong></p>
<ul>
<li><code>data</code>: source DataFrame.</li>
<li><code>index</code>: variables placed on rows.</li>
<li><code>columns</code>: variables placed across columns.</li>
<li><code>values</code>: numeric column(s) to summarize.</li>
<li><code>aggfunc</code>: aggregation such as <code>"sum"</code>, <code>"mean"</code>, or a list of functions.</li>
<li><code>fill_value</code>: replace missing cells in the result.</li>
<li><code>margins=True</code>: add row and column totals.</li>
<li><code>margins_name</code>: label for totals, such as <code>"Total"</code>.</li>
</ul>

<p><strong>pivot() vs pivot_table()</strong></p>
<ul>
<li><code>pivot()</code> reshapes but does not aggregate duplicate combinations.</li>
<li><code>pivot_table()</code> can aggregate multiple rows that belong to the same cell.</li>
</ul>
</details>

In [ ]:
report = pd.pivot_table(
    orders,
    index="region",
    columns="channel",
    values="amount",
    aggfunc="sum",
    fill_value=0
)

report

## 7. Reshape wide data with `melt()`

Tabular data is often stored in either **wide** or **long** form.

- **Wide format:** repeated measurements appear in separate columns.
- **Long format:** repeated measurements appear in rows, usually with one column identifying the variable and another containing the value.

Long format is often easier for grouping, plotting, and statistical workflows.

We first create a monthly summary, reshape it to a wide table with `pivot()`, and then return it to long format with `melt()`.

<details>
<summary><strong>Syntax & parameters — pivot() and melt()</strong></summary>

<p><strong>Wide format</strong></p>
<pre><code class="language-python">wide = df.pivot(
    index="region",
    columns="month",
    values="amount"
)</code></pre>

<p><strong>Long format</strong></p>
<pre><code class="language-python">long = wide.melt(
    id_vars="region",
    var_name="month",
    value_name="sales"
)</code></pre>

<p><strong>melt() parameters</strong></p>
<ul>
<li><code>id_vars</code>: columns that stay fixed as identifiers.</li>
<li><code>value_vars</code>: columns to unpivot; if omitted, Pandas uses all non-ID columns.</li>
<li><code>var_name</code>: name for the new variable-name column.</li>
<li><code>value_name</code>: name for the new values column.</li>
<li><code>ignore_index</code>: whether to create a fresh index.</li>
</ul>

<p>Related reshaping tools include <code>stack()</code>, <code>unstack()</code>, <code>pivot()</code>, and <code>pivot_table()</code>.</p>
</details>

In [ ]:
monthly_sales = (
    orders
    .assign(date=lambda df: pd.to_datetime(df["date"]))
    .assign(month=lambda df: df["date"].dt.strftime("%b"))
    .groupby(["region", "month"], as_index=False)["amount"]
    .sum()
)

wide = monthly_sales.pivot(
    index="region",
    columns="month",
    values="amount"
).fillna(0).reset_index()

wide

In [ ]:
long_sales = wide.melt(
    id_vars="region",
    var_name="month",
    value_name="sales"
)

long_sales.head(12)

## 8. One item per row with `explode()`

Sometimes a single cell contains multiple values.

The `tags` column stores values such as `student|learning`. That is convenient for storage, but not ideal for counting or grouping individual tags.

The workflow is:

1. split the string into a Python list;
2. use `explode()` to create one row per list item;
3. analyze the expanded values normally.

Columns that are not exploded are repeated automatically for each new row.

<details>
<summary><strong>Syntax & parameters — str.split() and explode()</strong></summary>

<p><strong>Split a delimited string</strong></p>
<pre><code class="language-python">df["tags"] = df["tags"].str.split("|")</code></pre>

<p>This changes a value such as <code>"student|learning"</code> into a Python list:</p>
<pre><code>["student", "learning"]</code></pre>

<p><strong>Explode the list</strong></p>
<pre><code class="language-python">df.explode("tags", ignore_index=False)</code></pre>

<ul>
<li><code>column</code>: list-like column to expand.</li>
<li><code>ignore_index=True</code>: create a new 0...N index.</li>
</ul>

<p><strong>Related string methods</strong></p>
<ul>
<li><code>.str.strip()</code></li>
<li><code>.str.lower()</code></li>
<li><code>.str.contains()</code></li>
<li><code>.str.replace()</code></li>
<li><code>.str.extract()</code></li>
</ul>

<p>After exploding, methods such as <code>value_counts()</code> and <code>groupby()</code> become much easier to use on individual items.</p>
</details>

In [ ]:
orders_tags = orders[["order_id", "product", "tags"]].copy()

orders_tags["tags"] = orders_tags["tags"].str.split("|")

orders_tags.head()

In [ ]:
tags_long = orders_tags.explode("tags")

tags_long.head(10)

In [ ]:
tags_long["tags"].value_counts()

## 9. Work with dates

Dates often arrive from CSV files as ordinary strings.

Convert them to Pandas datetime values before doing time-based analysis. Once a column has datetime dtype, the `.dt` accessor provides components such as:

- year;
- month;
- day;
- weekday;
- quarter;
- month name.

Correct datetime conversion is important because sorting, filtering, resampling, and date arithmetic depend on true datetime values rather than text.

<details>
<summary><strong>Syntax & parameters — to_datetime() and .dt</strong></summary>

<pre><code class="language-python">pd.to_datetime(
    df["date"],
    errors="coerce"
)</code></pre>

<p><strong>Useful to_datetime() parameters</strong></p>
<ul>
<li><code>errors="raise"</code>: stop on invalid dates.</li>
<li><code>errors="coerce"</code>: convert invalid dates to <code>NaT</code>.</li>
<li><code>format</code>: specify a known date format for stricter parsing.</li>
<li><code>dayfirst=True</code>: interpret ambiguous dates with day first.</li>
<li><code>utc=True</code>: return UTC-aware timestamps.</li>
</ul>

<p><strong>Useful .dt attributes</strong></p>
<pre><code class="language-python">df["date"].dt.year
df["date"].dt.month
df["date"].dt.month_name()
df["date"].dt.day
df["date"].dt.day_name()
df["date"].dt.quarter
df["date"].dt.to_period("M")</code></pre>

<p>Missing datetime values are represented as <code>NaT</code>, which means “Not a Time.”</p>
</details>

In [ ]:
orders_dates = orders.copy()
orders_dates["date"] = pd.to_datetime(orders_dates["date"])

orders_dates["weekday"] = orders_dates["date"].dt.day_name()
orders_dates["month"] = orders_dates["date"].dt.to_period("M")
orders_dates["day"] = orders_dates["date"].dt.day

orders_dates[["order_id", "date", "weekday", "month", "day"]]

## 10. Time-based summaries with `resample()`

`resample()` is similar to `groupby()`, but the groups are regular time intervals.

Examples include:

- daily totals;
- weekly averages;
- monthly counts;
- quarterly summaries.

The datetime information can be in the index or supplied through the `on=` parameter.

Here we calculate daily sales totals from the `date` column.

<details>
<summary><strong>Syntax & parameters — resample()</strong></summary>

<pre><code class="language-python">df.resample(
    "D",
    on="date"
)["amount"].sum()</code></pre>

<p>The first argument is a frequency rule.</p>

<p><strong>Common rules</strong></p>
<ul>
<li><code>"D"</code> — daily</li>
<li><code>"W"</code> — weekly</li>
<li><code>"MS"</code> — month start</li>
<li><code>"ME"</code> — month end</li>
<li><code>"QS"</code> — quarter start</li>
<li><code>"YE"</code> — year end</li>
</ul>

<p><strong>Useful parameters</strong></p>
<ul>
<li><code>on</code>: datetime column when the index is not datetime.</li>
<li><code>label</code>: choose which interval boundary labels each group.</li>
<li><code>closed</code>: choose which interval boundary is included.</li>
<li><code>origin</code>: control alignment of time bins.</li>
</ul>

<p>After resampling, use an aggregation such as <code>sum()</code>, <code>mean()</code>, <code>count()</code>, or <code>agg()</code>.</p>
</details>

In [ ]:
daily_sales = (
    orders_dates
    .resample("D", on="date")["amount"]
    .sum()
)

daily_sales

### Try it
Change `"D"` to `"W"` and compare the result.

## 11. Rolling summaries

A **rolling window** calculates a statistic over a moving group of neighboring observations.

For a 3-row rolling mean:

- row 1 does not yet have three values;
- row 2 still does not have three values;
- row 3 uses rows 1–3;
- row 4 uses rows 2–4;
- and so on.

Rolling summaries are useful for smoothing short-term variation and studying local trends.

They do not replace the original values; they provide another analytical view.

<details>
<summary><strong>Syntax & parameters — rolling()</strong></summary>

<pre><code class="language-python">series.rolling(
    window=3,
    min_periods=None,
    center=False
).mean()</code></pre>

<p><strong>Main parameters</strong></p>
<ul>
<li><code>window</code>: number of rows, or a time window such as <code>"7D"</code> when using a datetime index.</li>
<li><code>min_periods</code>: minimum observations required before a value is returned.</li>
<li><code>center=True</code>: place the window label at the center instead of the right edge.</li>
</ul>

<p><strong>Common rolling calculations</strong></p>
<pre><code class="language-python">s.rolling(7).mean()
s.rolling(7).sum()
s.rolling(7).median()
s.rolling(7).std()</code></pre>

<p>To avoid initial missing values:</p>
<pre><code class="language-python">s.rolling(window=3, min_periods=1).mean()</code></pre>

<p>Be careful not to use future observations when building predictive features from time-series data.</p>
</details>

In [ ]:
daily = (
    orders_dates
    .resample("D", on="date")["amount"]
    .sum()
    .reset_index(name="sales")
)

daily["rolling_3"] = (
    daily["sales"]
    .rolling(window=3)
    .mean()
)

daily.head(10).round(2)

The first two rolling values are missing because a complete 3-row window does not exist yet.

## 12. Create categories with `map()`, `replace()`, and `cut()`

Pandas provides several ways to recode values.

- `map()` is convenient for mapping individual values to new labels.
- `replace()` substitutes selected values or patterns.
- `cut()` converts continuous numeric values into intervals.

These functions solve related but different problems. Choosing the right one makes recoding easier to read and maintain.

<details>
<summary><strong>Syntax & parameters — map(), replace(), and cut()</strong></summary>

<p><strong>map()</strong></p>
<pre><code class="language-python">df["channel"].map({
    "Web": "Online",
    "Store": "In person"
})</code></pre>

<p><code>map()</code> can accept a dictionary, Series, or function. Unmapped dictionary values normally become <code>NaN</code>.</p>

<p><strong>replace()</strong></p>
<pre><code class="language-python">df["region"].replace({
    "North": "N",
    "South": "S"
})</code></pre>

<p><code>replace()</code> leaves unmatched values unchanged. It can also work with lists and regular expressions.</p>

<p><strong>cut()</strong></p>
<pre><code class="language-python">pd.cut(
    df["amount"],
    bins=[0, 60, 100, float("inf")],
    labels=["Small", "Medium", "Large"],
    right=True,
    include_lowest=True
)</code></pre>

<ul>
<li><code>bins</code>: interval edges.</li>
<li><code>labels</code>: category names.</li>
<li><code>right</code>: whether the right edge is included.</li>
<li><code>include_lowest</code>: include the lowest boundary.</li>
<li><code>duplicates</code>: behavior when bin edges are duplicated.</li>
</ul>

<p>Related function: <code>pd.qcut()</code> creates bins based on quantiles rather than fixed numeric boundaries.</p>
</details>

In [ ]:
orders_categories = orders.copy()

orders_categories["channel_label"] = orders_categories["channel"].map({
    "Web": "Online",
    "Email": "Online",
    "Store": "In person"
})

orders_categories["order_size"] = pd.cut(
    orders_categories["amount"],
    bins=[0, 60, 100, float("inf")],
    labels=["Small", "Medium", "Large"]
)

orders_categories[
    ["order_id", "channel", "channel_label", "amount", "order_size"]
]

In [ ]:
# A small replace() example
orders_categories["region_short"] = orders_categories["region"].replace({
    "North": "N",
    "South": "S",
    "West": "W"
})

orders_categories[["region", "region_short"]].drop_duplicates()

## 13. Safer joins with `merge()`

A join combines columns from related tables using one or more key columns.

The difficult part is often not the syntax—it is verifying that the key relationship is what you expect.

Two parameters are especially useful:

- `validate=` checks whether the join has the expected one-to-one or many-to-one structure;
- `indicator=True` adds a `_merge` column so you can see which rows matched.

Here we create a small customer lookup table and merge customer attributes back onto the order table.

<details>
<summary><strong>Syntax & parameters — merge()</strong></summary>

<pre><code class="language-python">left.merge(
    right,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator=True
)</code></pre>

<p><strong>Join types</strong></p>
<ul>
<li><code>how="left"</code> — keep every row from the left table.</li>
<li><code>how="inner"</code> — keep only matching keys.</li>
<li><code>how="right"</code> — keep every row from the right table.</li>
<li><code>how="outer"</code> — keep keys from both tables.</li>
</ul>

<p><strong>Key parameters</strong></p>
<ul>
<li><code>on</code>: same key name in both tables.</li>
<li><code>left_on</code> / <code>right_on</code>: different key names.</li>
<li><code>suffixes</code>: labels overlapping non-key columns.</li>
<li><code>validate</code>: check expected relationship.</li>
<li><code>indicator=True</code>: add <code>_merge</code> showing match source.</li>
</ul>

<p><strong>validate values</strong></p>
<ul>
<li><code>"one_to_one"</code></li>
<li><code>"one_to_many"</code></li>
<li><code>"many_to_one"</code></li>
<li><code>"many_to_many"</code></li>
</ul>

<p>A successful merge is not automatically a correct merge. Always check row counts, unmatched keys, and duplicate keys.</p>
</details>

In [ ]:
orders_for_merge = orders[
    ["order_id", "customer_id", "amount"]
].copy()

# Ensure customer_id is strictly unique by using subset
customers = (
    orders[
        ["customer_id", "customer_segment"]
    ]
    .drop_duplicates(subset="customer_id")
)

merged = orders_for_merge.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

merged.head()

In [ ]:
# Check whether any order failed to match a customer.
merged["_merge"].value_counts()

## 14. Reusable workflows with `pipe()`

`pipe()` passes a DataFrame into a function.

This helps separate a long workflow into small named transformations.

Instead of embedding every transformation directly inside one large chain, you can write focused functions such as:

- `clean_columns()`;
- `add_order_size()`;
- `filter_valid_orders()`;
- `create_report()`.

A well-named function can make a chain read almost like a sentence.

<details>
<summary><strong>Syntax & parameters — pipe()</strong></summary>

<pre><code class="language-python">df.pipe(function_name)</code></pre>

<p>The DataFrame is passed as the first argument to the function.</p>

<pre><code class="language-python">def keep_large_orders(df, minimum):
    return df.loc[df["amount"] &gt;= minimum]

result = (
    orders
    .pipe(keep_large_orders, minimum=100)
)</code></pre>

<p><strong>Why pipe?</strong></p>
<ul>
<li>keeps a chain readable;</li>
<li>lets you reuse tested transformation functions;</li>
<li>makes complex workflows easier to organize;</li>
<li>reduces deeply nested function calls.</li>
</ul>

<p>A function used with <code>pipe()</code> should normally return a DataFrame or Series if you want the Pandas chain to continue.</p>
</details>

In [ ]:
def add_order_size(df):
    return df.assign(
        order_size=pd.cut(
            df["amount"],
            bins=[0, 60, 100, float("inf")],
            labels=["Small", "Medium", "Large"]
        )
    )


pipe_result = (
    orders
    .pipe(add_order_size)
    .sort_values("amount", ascending=False)
)

pipe_result[["order_id", "amount", "order_size"]]

## 15. Mini workflow: raw orders to report

This section combines several ideas into one end-to-end analytical workflow.

The chain:

1. converts the date;
2. removes non-positive amounts;
3. creates a monthly period;
4. groups by month and region;
5. calculates sales and order counts;
6. sorts the report.

The important skill is not memorizing this exact chain. It is learning how to organize several transformations so another person can follow the logic from raw data to final report.

<details>
<summary><strong>Syntax guide — building an end-to-end chain</strong></summary>

<p>A useful way to design a long workflow is to think in stages:</p>

<pre><code class="language-python">report = (
    df
    # 1. standardize types
    .assign(...)

    # 2. keep valid rows
    .query(...)

    # 3. create analytical features
    .assign(...)

    # 4. summarize
    .groupby(..., as_index=False)
    .agg(...)

    # 5. organize final output
    .sort_values(...)
)</code></pre>

<p><strong>Questions to ask at each stage</strong></p>
<ul>
<li>Did the row count change? Should it have?</li>
<li>Did a data type change?</li>
<li>Were missing values introduced?</li>
<li>Are grouping keys correct?</li>
<li>Are the aggregation units clear?</li>
<li>Does the final sort make the report easier to read?</li>
</ul>

<p>For complex pipelines, save or inspect intermediate results while developing the code. A single chain should be readable, not mysterious.</p>
</details>

In [ ]:
final_report = (
    orders
    .assign(date=lambda df: pd.to_datetime(df["date"]))
    .query("amount > 0")
    .assign(month=lambda df: df["date"].dt.to_period("M"))
    .groupby(["month", "region"], as_index=False)
    .agg(
        sales=("amount", "sum"),
        orders=("order_id", "count")
    )
    .sort_values(["month", "sales"], ascending=[True, False])
)

final_report

## 16. Practice challenge

Use the CSV-backed `orders` DataFrame to build your own workflow.

Your goal is to practice selecting the correct Pandas methods and connecting them in a readable order.

Build a summary that:

1. converts `date` to datetime;
2. keeps orders with `amount >= 50`;
3. creates an `order_size` category:
   - Small: 0–60
   - Medium: 60–100
   - Large: above 100
4. groups by `region`;
5. calculates:
   - `total_sales`
   - `average_order`
   - `orders`
6. sorts from highest to lowest `total_sales`.

Try to complete the workflow before opening the syntax guide below.

<details>
<summary><strong>Hints & syntax — practice challenge</strong></summary>

<p>Useful methods for the challenge:</p>
<pre><code class="language-python">pd.to_datetime(...)
df.query(...)
df.assign(...)
pd.cut(...)
df.groupby(..., as_index=False)
df.agg(...)
df.sort_values(...)</code></pre>

<p><strong>Suggested structure</strong></p>
<pre><code class="language-python">practice = (
    orders
    .assign(...)
    .query(...)
    .assign(...)
    .groupby(..., as_index=False)
    .agg(...)
    .sort_values(...)
)</code></pre>

<p>Try writing each step yourself before comparing it with earlier examples.</p>
</details>

In [ ]:
# Write your practice solution here.

practice = orders.copy()

# Continue below:

## Summary and self-check

Advanced Pandas becomes most useful when several operations are combined into a clear analytical workflow.

Check that you can explain:

- why method chaining can improve readability;
- when `transform()` is preferable to an aggregation;
- how wide and long data differ;
- what `melt()` and `explode()` change;
- why datetime conversion matters before resampling;
- what a rolling summary represents;
- why merge validation can catch data problems;
- how `pipe()` supports reusable transformations.

A strong workflow should be understandable from top to bottom without requiring hidden manual steps.